In [20]:
#Importing the cleaned dataset
import pandas as pd
df = pd.read_csv('final_dataset.csv', keep_default_na=False, na_values=[''])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 53817 entries, 0 to 53816
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   REF_AREA              53817 non-null  int64  
 1   TIME_PERIOD           53817 non-null  int64  
 2   MUNICIPALITY          53817 non-null  str    
 3   PROVINCE_CODE         53817 non-null  str    
 4   PROVINCE_NAME         53817 non-null  str    
 5   SURFACE_KMQ           53817 non-null  float64
 6   SURFACE_YEAR          53817 non-null  int64  
 7   POPULATION            53817 non-null  int64  
 8   POPULATION_YEAR       53817 non-null  int64  
 9   INJURED               53817 non-null  int64  
 10  KILLED                53817 non-null  int64  
 11  TOTAL_ACCIDENTS       53817 non-null  int64  
 12  ACCIDENTS_PER_CAPITA  53817 non-null  float64
 13  ACCIDENTS_PER_KMQ     53817 non-null  float64
dtypes: float64(3), int64(8), str(3)
memory usage: 5.7 MB


In [21]:
#Checking basic statistics on the new columns to find outliers
df[['ACCIDENTS_PER_CAPITA', 'ACCIDENTS_PER_KMQ']].describe()

,ACCIDENTS_PER_CAPITA,ACCIDENTS_PER_KMQ
count,53817.000000,53817.000000
mean,17.654290,0.617001
std,19.016195,1.607034
min,0.000000,0.000000
25%,5.420054,0.029524
50%,14.362313,0.149447
75%,24.673671,0.517810
max,851.063830,46.912592


In [22]:
# Investigating the outliers

print(df[df['ACCIDENTS_PER_CAPITA'] == df['ACCIDENTS_PER_CAPITA'].max()][['MUNICIPALITY', 'TIME_PERIOD', 'POPULATION', 'TOTAL_ACCIDENTS', 'ACCIDENTS_PER_CAPITA']])
print(df[df['ACCIDENTS_PER_KMQ'] == df['ACCIDENTS_PER_KMQ'].max()][['MUNICIPALITY', 'TIME_PERIOD', 'SURFACE_KMQ', 'TOTAL_ACCIDENTS', 'ACCIDENTS_PER_KMQ']])

     MUNICIPALITY  TIME_PERIOD  POPULATION  TOTAL_ACCIDENTS  \
1038   Moncenisio         2022          47                4   

      ACCIDENTS_PER_CAPITA  
1038             851.06383  
      MUNICIPALITY  TIME_PERIOD  SURFACE_KMQ  TOTAL_ACCIDENTS  \
11931       Milano         2018     181.6783             8523   

       ACCIDENTS_PER_KMQ  
11931          46.912592  


In [23]:
# Top 10 rows by ACCIDENTS_PER_CAPITA
df.sort_values('ACCIDENTS_PER_CAPITA', ascending=False).head(10)[['MUNICIPALITY', 'PROVINCE_NAME', 'TIME_PERIOD', 'POPULATION', 'TOTAL_ACCIDENTS', 'ACCIDENTS_PER_CAPITA']]

,MUNICIPALITY,PROVINCE_NAME,TIME_PERIOD,POPULATION,TOTAL_ACCIDENTS,ACCIDENTS_PER_CAPITA
1038,Moncenisio,Torino,2022,47,4,851.063830
18916,Castel Condino,Trento,2019,224,11,491.071429
18359,Ponte Gardena/Waidbruck,Bolzano/Bozen,2018,195,8,410.256410
18915,Castel Condino,Trento,2018,222,9,405.405405
3290,Argentera,Cuneo,2024,77,3,389.610390
6937,Bard,Valle d'Aosta/Vallée d'Aoste,2019,122,4,327.868852
50498,Morterone,Lecco,2023,33,1,303.030303
1034,Moncenisio,Torino,2018,33,1,303.030303
6941,Bard,Valle d'Aosta/Vallée d'Aoste,2023,102,3,294.117647
50079,Villanova Biellese,Biella,2023,175,5,285.714286


In [24]:
# Top 10 rows by ACCIDENTS_PER_KMQ
df.sort_values('ACCIDENTS_PER_KMQ', ascending=False).head(10)[['MUNICIPALITY', 'PROVINCE_NAME', 'TIME_PERIOD', 'SURFACE_KMQ', 'TOTAL_ACCIDENTS', 'ACCIDENTS_PER_KMQ']]

,MUNICIPALITY,PROVINCE_NAME,TIME_PERIOD,SURFACE_KMQ,TOTAL_ACCIDENTS,ACCIDENTS_PER_KMQ
11931,Milano,Milano,2018,181.6783,8523,46.912592
11932,Milano,Milano,2019,181.6783,8263,45.481491
11936,Milano,Milano,2023,181.8450,7812,42.959663
11935,Milano,Milano,2022,181.8450,7783,42.800187
11937,Milano,Milano,2024,181.8450,7743,42.580219
11934,Milano,Milano,2021,181.8450,7465,41.051445
11734,Cormano,Milano,2024,4.4733,126,28.167125
11698,Cinisello Balsamo,Milano,2023,12.7240,353,27.742848
12214,Sesto San Giovanni,Milano,2022,11.6989,323,27.609433
11699,Cinisello Balsamo,Milano,2024,12.7240,348,27.349890


In [28]:
#Capping outliers above Q3 + 1.5*IQR for each column, keeping originals for reference
columns_to_check = ['TOTAL_ACCIDENTS', 'ACCIDENTS_PER_CAPITA', 'ACCIDENTS_PER_KMQ']

for column_name in columns_to_check:
    q1 = df[column_name].quantile(0.25)
    q3 = df[column_name].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    capped_column_name = 'CAPPED_' + column_name
    df[capped_column_name] = df[column_name].clip(upper=upper_bound)

    outlier_count = (df[column_name] > upper_bound).sum()
    outlier_percentage = round(outlier_count / len(df) * 100, 2)
    print(column_name, '- outliers:', outlier_count, '(', outlier_percentage, '%)')


TOTAL_ACCIDENTS - outliers: 6616 ( 12.29 %)
ACCIDENTS_PER_CAPITA - outliers: 1874 ( 3.48 %)
ACCIDENTS_PER_KMQ - outliers: 6354 ( 11.81 %)
